# Crimes of India: Regional Analysis, 2014–2016

This notebook examines recorded Indian Penal Code (IPC) crimes across India’s States and Union Territories using the project’s supplied table. It considers recorded counts, population-adjusted rates, regional changes, and differences between State and UT rate distributions.

> **Scope and interpretation:** Recorded crimes reflect incidents in the reporting system; they are not necessarily the true incidence of all crimes committed. Counts and rates answer different questions and are affected by reporting practices and population differences.

### Research questions

- How did recorded IPC crime totals change from 2014 to 2016?
- Which regions had the highest counts and highest or lowest rates in 2016?
- Which regions experienced the largest count increases and decreases?
- How do crime-rate distributions compare across States and UTs?
- Is projected population linearly associated with crime rate across the 36 regions?

**Source:** `data/raw/Table_1A.1_2016.csv`  
**Period:** 2014–2016  
**Unit of analysis:** 29 States and 7 Union Territories.

## 2. Load and validate the data

The source table includes three aggregate rows—Total States, Total UTs, and Total All-India—in addition to individual regions. We exclude those summary rows before calculating regional results, which avoids counting the same crimes once in regional records and again in aggregate rows. The notebook reads the raw data and independently checks its cleaned records.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from IPython import get_ipython
get_ipython().run_line_magic("matplotlib", "inline")

plt.rcParams.update({"font.size": 10, "axes.titlesize": 13})
RAW_RELATIVE_PATH = Path("data") / "raw" / "Table_1A.1_2016.csv"

def find_project_root(start=None):
    """Find the project from the current folder or its parents."""
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / RAW_RELATIVE_PATH).is_file():
            return candidate
    raise FileNotFoundError(
        f"Could not find {RAW_RELATIVE_PATH}; open Jupyter from the project or notebooks folder."
    )

PROJECT_DIR = find_project_root()
print(f"Notebook kernel interpreter: {sys.executable}")
project_venv = (PROJECT_DIR / ".venv").resolve()
active_venv = Path(sys.prefix).resolve()
if project_venv.is_dir():
    assert active_venv == project_venv, "Select this project’s .venv kernel before running the notebook."
else:
    assert active_venv != Path(sys.base_prefix).resolve(), "Select a Python virtual environment for this notebook."
    print(f"Using virtual environment: {active_venv}")
DATA_PATH = PROJECT_DIR / RAW_RELATIVE_PATH
raw_df = pd.read_csv(DATA_PATH)
print(f"Project folder: {PROJECT_DIR}\nRaw file: {DATA_PATH}\nRaw shape: {raw_df.shape}")
display(pd.DataFrame({"column": raw_df.columns, "dtype": raw_df.dtypes.astype(str).values}))
display(raw_df.head())

In [ ]:
EXPECTED_COLUMNS = [
    "serial_no", "category", "state_ut", "crimes_2014", "crimes_2015",
    "crimes_2016", "percentage_share_2016", "incidence_rank_2016",
    "population_lakhs_2016", "crime_rate_2016", "crime_rate_rank_2016",
]
assert raw_df.shape[1] == len(EXPECTED_COLUMNS), f"Expected 11 columns, found {raw_df.shape[1]}"

serial_values = pd.to_numeric(raw_df.iloc[:, 0], errors="coerce")
excluded_rows = raw_df.loc[serial_values.isna()]
expected_summary_names = {"Total (States)", "Total (UTs)", "Total (All-India)"}
excluded_names = set(excluded_rows.iloc[:, 2].astype(str).str.strip())
assert excluded_names == expected_summary_names, f"Unexpected excluded rows: {excluded_names}"

crime = raw_df.loc[serial_values.notna()].copy()
crime.columns = EXPECTED_COLUMNS
crime["serial_no"] = pd.to_numeric(crime["serial_no"], errors="raise").astype(int)
crime["category"] = crime["category"].astype(str).str.strip()
crime["state_ut"] = crime["state_ut"].astype(str).str.strip()
for column in EXPECTED_COLUMNS[3:]:
    crime[column] = pd.to_numeric(crime[column], errors="coerce")

missing = crime.isna().sum()
quality = pd.Series({
    "region records": len(crime),
    "duplicate serial numbers": int(crime["serial_no"].duplicated().sum()),
    "duplicate region names": int(crime["state_ut"].duplicated().sum()),
    "invalid categories": int((~crime["category"].isin(["State", "UT"])).sum()),
    "negative count/population/rate values": int((crime[["crimes_2014", "crimes_2015", "crimes_2016", "population_lakhs_2016", "crime_rate_2016"]] < 0).sum().sum()),
    "missing values": int(missing.sum()),
})
display(quality.to_frame("value"))
display(missing.rename("missing values").to_frame())
assert len(crime) == 36
assert missing.sum() == 0
assert crime["serial_no"].is_unique and crime["state_ut"].is_unique
assert set(crime["category"]) == {"State", "UT"}
assert (crime[["crimes_2014", "crimes_2015", "crimes_2016", "population_lakhs_2016", "crime_rate_2016"]] >= 0).all().all()
print(f"Validation passed: {len(crime)} regions, {(crime.category == 'State').sum()} States, {(crime.category == 'UT').sum()} UTs. Excluded the three aggregate rows.")

# Compare the generated cleaned file, if present, with the raw-source analysis.
processed_path = PROJECT_DIR / "processed" / "cleaned_crime_data.csv"
if processed_path.is_file():
    processed = pd.read_csv(processed_path)
    assert len(processed) == len(crime)
    assert processed["state_ut"].tolist() == crime["state_ut"].tolist()
    for year in (2014, 2015, 2016):
        column = f"crimes_{year}"
        assert np.array_equal(processed[column].to_numpy(), crime[column].to_numpy())
    print("Processed CSV check passed: regional records and annual counts match the raw source.")
else:
    print("No processed CSV was found; the notebook continues from the raw source.")

### Cleaning decisions

The source uses a serial number to distinguish region records from summary rows. Rows with a nonnumeric serial are excluded, and the three expected aggregate labels are checked explicitly. The 11 source columns are renamed to concise analysis names; numeric fields are converted with invalid values becoming missing, then checked. No data is imputed.

## 3. Exploratory data analysis

The first table describes **regional counts** for each year. `50%` is the median; `Q1` and `Q3` are the first and third quartiles; the IQR is `Q3 − Q1`. These are unweighted summaries across regions.

In [ ]:
count_columns = ["crimes_2014", "crimes_2015", "crimes_2016"]
count_summary = crime[count_columns].describe().T.rename(
    columns={"25%": "Q1", "50%": "median", "75%": "Q3"}
)
count_summary["IQR"] = count_summary["Q3"] - count_summary["Q1"]
display(count_summary[["count", "mean", "median", "std", "min", "Q1", "Q3", "IQR", "max"]].round(2))

Crime-rate outliers are statistical flags for follow-up, not evidence that a value is wrong. The 1.5-IQR fences summarize the overall 2016 rate distribution.

In [ ]:
rate = crime["crime_rate_2016"]
rate_q1, rate_q3 = rate.quantile([0.25, 0.75])
rate_iqr = rate_q3 - rate_q1
rate_lower_fence = rate_q1 - 1.5 * rate_iqr
rate_upper_fence = rate_q3 + 1.5 * rate_iqr
rate_outliers = crime.loc[
    (rate < rate_lower_fence) | (rate > rate_upper_fence),
    ["state_ut", "category", "crime_rate_2016"],
].sort_values("crime_rate_2016", ascending=False)
print(f"Fences: {rate_lower_fence:.1f} to {rate_upper_fence:.1f} crimes per 100,000 population")
display(rate_outliers.reset_index(drop=True))

## 4. Overall crime trends

Totals are computed from the 36 regional records. The known totals are used as validation checks—not as substitutes for calculating the results. 2014 is the baseline year because the source contains no 2013 total.

In [ ]:
year_columns = {2014: "crimes_2014", 2015: "crimes_2015", 2016: "crimes_2016"}
yearly_totals = crime[list(year_columns.values())].sum()
yearly_totals.index = list(year_columns)
yearly_change = yearly_totals.diff()
yearly_percent_change = yearly_totals.pct_change() * 100
trend_table = pd.DataFrame({
    "total_recorded_crimes": yearly_totals,
    "change_from_previous_year": yearly_change,
    "percentage_change_from_previous_year": yearly_percent_change,
})
display(trend_table.round(2))

reference_totals = pd.Series({2014: 2_851_563, 2015: 2_949_400, 2016: 2_975_711})
pd.testing.assert_series_equal(yearly_totals.astype("int64"), reference_totals, check_names=False)
overall_absolute_change = int(yearly_totals.loc[2016] - yearly_totals.loc[2014])
overall_percentage_change = (yearly_totals.loc[2016] / yearly_totals.loc[2014] - 1) * 100
print(f"Overall 2014–2016 change: {overall_absolute_change:+,} crimes ({overall_percentage_change:+.2f}%).")
print("Calculated totals match all three reference totals.")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(yearly_totals.index, yearly_totals.values, marker="o", linewidth=2.5, color="#245b8f")
ax.set(title="Total Recorded IPC Crimes Across States/UTs (2014–2016)",
       xlabel="Year", ylabel="Recorded IPC crimes (count)")
ax.set_xticks([2014, 2015, 2016])
ax.set_ylim(0, yearly_totals.max() * 1.08)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()

## 5. Crime counts versus crime rates

A count answers where the most crimes were recorded; it is affected by regional population size. A population-adjusted rate answers a different question. The source defines its 2016 crime rate per 100,000 population. Neither measure alone explains why regions differ.

In [1]:
top_counts = crime.nlargest(10, "crimes_2016")[["state_ut", "category", "crimes_2016", "crime_rate_2016"]]
highest_rates = crime.nlargest(10, "crime_rate_2016")[["state_ut", "category", "crime_rate_2016", "crimes_2016"]]
lowest_rates = crime.nsmallest(10, "crime_rate_2016")[["state_ut", "category", "crime_rate_2016", "crimes_2016"]]
print("Top 10 2016 crime counts; count and rate remain separate measures:")
display(top_counts.reset_index(drop=True))
print("Highest 10 rates (per 100,000 population):")
display(highest_rates.reset_index(drop=True))
print("Lowest 10 rates (per 100,000 population):")
display(lowest_rates.reset_index(drop=True))

NameError: name 'crime' is not defined

In [ ]:
ranked_counts = top_counts.sort_values("crimes_2016")
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(ranked_counts["state_ut"], ranked_counts["crimes_2016"], color="#3977a8")
ax.set(title="Top 10 States/UTs by Recorded IPC Crimes (2016)",
       xlabel="Recorded IPC crimes (count)", ylabel="State/UT")
ax.grid(axis="x", alpha=0.22)
ax.set_axisbelow(True)
fig.tight_layout()
plt.show()

### Share of recorded crime by region

This donut chart shows how each region contributes to the 2016 recorded-crime total. The eight largest regions are shown separately and the remaining 28 are combined as “Others.” Wedge sizes represent each group’s share of the total; the legend identifies each group.

In [ ]:
ranked_2016 = crime.sort_values("crimes_2016", ascending=False)
leading_regions = ranked_2016.head(8).set_index("state_ut")["crimes_2016"]
other_regions = ranked_2016.iloc[8:]["crimes_2016"].sum()
share_values = leading_regions.tolist() + [other_regions]
share_labels = leading_regions.index.tolist() + ["Others (remaining regions)"]

fig, ax = plt.subplots(figsize=(9, 6.5))
wedges, _, _ = ax.pie(
    share_values,
    startangle=90,
    counterclock=False,
    autopct=lambda pct: f"{pct:.1f}%" if pct >= 4 else "",
    pctdistance=0.78,
    wedgeprops={"width": 0.42, "edgecolor": "white", "linewidth": 1.2},
    colors=plt.get_cmap("Blues")(np.linspace(0.35, 0.9, len(share_values))),
)
ax.text(0, 0.08, "2016", ha="center", va="center", fontsize=15, weight="bold")
ax.text(0, -0.16, "recorded crimes", ha="center", va="center", fontsize=9)
ax.set_title("Share of Recorded IPC Crimes by State/UT (2016)", pad=16)
ax.legend(wedges, share_labels, title="Region", loc="center left",
          bbox_to_anchor=(1.0, 0.5), frameon=False)
fig.text(0.5, 0.025,
         "Percentages show each region’s share of the 36-region total; Others combines the remaining 28.",
         ha="center", fontsize=9)
fig.tight_layout(rect=[0, 0.06, 0.78, 1])
plt.show()

## 6. Regional changes, 2014–2016

Absolute change is `2016 count − 2014 count`. Percentage change is that difference relative to the 2014 count. Absolute change shows the difference in recorded counts; relative change shows its size compared with the baseline. A zero baseline has no defined percentage change and is returned as `NaN` rather than zero.

In [ ]:
change = crime["crimes_2016"] - crime["crimes_2014"]
baseline = crime["crimes_2014"].to_numpy(dtype=float)
percentage_change = np.divide(
    change.to_numpy(dtype=float) * 100,
    baseline,
    out=np.full(len(crime), np.nan, dtype=float),
    where=baseline != 0,
)
regional_changes = crime[["state_ut", "category", "crimes_2014", "crimes_2016"]].copy()
regional_changes["absolute_change"] = change
regional_changes["percentage_change"] = percentage_change
largest_increases = regional_changes.nlargest(10, "absolute_change")
largest_decreases = regional_changes.nsmallest(10, "absolute_change")
print("Largest increases:")
display(largest_increases[["state_ut", "category", "absolute_change", "percentage_change"]].reset_index(drop=True).round(2))
print("Largest decreases:")
display(largest_decreases[["state_ut", "category", "absolute_change", "percentage_change"]].reset_index(drop=True).round(2))

In [ ]:
extremes = pd.concat([largest_increases, largest_decreases]).drop_duplicates("state_ut").sort_values("absolute_change")
colors = ["#c44e52" if value < 0 else "#4c956c" for value in extremes["absolute_change"]]
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(extremes["state_ut"], extremes["absolute_change"], color=colors)
ax.axvline(0, color="#333333", linewidth=1)
ax.set(title="Largest State/UT Changes in Recorded IPC Crimes (2014–2016)",
       xlabel="Change in recorded crime count (2016 minus 2014)", ylabel="State/UT")
ax.grid(axis="x", alpha=0.22)
ax.set_axisbelow(True)
fig.tight_layout()
plt.show()

## 7. States versus Union Territories

The table and box plot compare the **distribution of regional crime rates**, not group crime totals. Each region contributes one value; summaries are not population-weighted. UTs have seven observations compared with 29 States, and the groups also differ in context, so group differences should be interpreted cautiously.

In [ ]:
rate_summary = crime.groupby("category")["crime_rate_2016"].agg(
    region_count="count", mean="mean", median="median", standard_deviation="std",
    minimum="min", maximum="max",
)
rate_summary["Q1"] = crime.groupby("category")["crime_rate_2016"].quantile(0.25)
rate_summary["Q3"] = crime.groupby("category")["crime_rate_2016"].quantile(0.75)
rate_summary["IQR"] = rate_summary["Q3"] - rate_summary["Q1"]
display(rate_summary[["region_count", "mean", "median", "standard_deviation", "minimum", "Q1", "Q3", "IQR", "maximum"]].round(2))

state_rates = crime.loc[crime["category"] == "State", "crime_rate_2016"]
ut_rates = crime.loc[crime["category"] == "UT", "crime_rate_2016"]
fig, ax = plt.subplots(figsize=(8, 5.5))
box = ax.boxplot([state_rates, ut_rates],
                 tick_labels=[f"States (n={len(state_rates)})", f"UTs (n={len(ut_rates)})"],
                 patch_artist=True, medianprops={"color": "#a63603", "linewidth": 2})
for patch, color in zip(box["boxes"], ["#9ecae1", "#fdae6b"]):
    patch.set_facecolor(color)
ax.set(title="2016 Recorded IPC Crime-Rate Distributions: States and UTs",
       ylabel="Recorded IPC crime rate (per 100,000 population)")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()

### 2016 crime-rate ranking by region

This lollipop chart displays the recorded IPC crime rate for every State and Union Territory. The horizontal axis is the rate per 100,000 population; the vertical axis lists regions in ascending order. Each dot marks a region’s rate, and its line from zero makes comparisons in position and magnitude easier. This complements the box plot by showing individual regional values.

In [ ]:
rate_ranking = crime.dropna(subset=["crime_rate_2016"]).sort_values("crime_rate_2016")
fig, ax = plt.subplots(figsize=(10, 12))
ax.hlines(rate_ranking["state_ut"], 0, rate_ranking["crime_rate_2016"],
          color="#9ecae1", linewidth=1.5)
ax.scatter(rate_ranking["crime_rate_2016"], rate_ranking["state_ut"],
           color="#245b8f", s=28, zorder=3)
ax.set(title="Recorded IPC Crime-Rate Ranking by State/UT (2016)",
       xlabel="Recorded IPC crime rate (per 100,000 population)",
       ylabel="State/UT")
ax.grid(axis="x", alpha=0.22)
ax.set_axisbelow(True)
fig.tight_layout()
plt.show()

## 8. Population and crime rate

Pearson’s *r* measures the strength and direction of a linear association across the regional observations. It does not establish causation. Population size alone does not account for reporting practices, demographic composition, or other factors that can affect recorded rates.

In [ ]:
population_rate = crime[["population_lakhs_2016", "crime_rate_2016"]].dropna()
pearson_r = population_rate["population_lakhs_2016"].corr(population_rate["crime_rate_2016"], method="pearson")
print(f"Pearson correlation (n={len(population_rate)} regions): r = {pearson_r:.3f}")
if abs(pearson_r) < 0.1:
    print("Interpretation: very weak linear association in this dataset; descriptive, not causal evidence.")

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(population_rate["population_lakhs_2016"], population_rate["crime_rate_2016"],
           s=55, alpha=0.8, color="#3977a8", edgecolor="white", linewidth=0.5)
ax.set(title="Projected Population vs Recorded IPC Crime Rate by State/UT (2016)",
       xlabel="Projected population (lakhs)", ylabel="Recorded IPC crime rate (per 100,000 population)")
ax.text(0.03, 0.97, f"Pearson r = {pearson_r:.3f}\nAssociation does not imply causation",
        transform=ax.transAxes, va="top", ha="left",
        bbox={"boxstyle": "round,pad=.35", "facecolor": "white", "alpha": 0.9, "edgecolor": "#cccccc"})
ax.grid(alpha=0.22)
fig.tight_layout()
plt.show()
assert np.isclose(pearson_r, 0.07513547772808332, atol=1e-12)

## 9. Visual analysis guide

The notebook contains seven charts placed beside the analysis they support: a line chart for yearly totals, a horizontal bar chart for the top count ranking, a donut chart for crime shares, a diverging bar chart for positive and negative count changes, a State/UT box plot for rate distributions, a lollipop chart for individual regional rate rankings, and a scatter plot for population and rate. The donut groups the 28 regions outside the top eight as “Others.” Tables retain the detailed high and low rate rankings and other summaries. IQR flags are shown with their values and should prompt context—not be treated as errors.

## 10. Findings and conclusion

The summary below reports observations calculated in the preceding cells.

In [ ]:
observations = pd.DataFrame({
    "measure": ["2014 total", "2015 total", "2016 total", "2014–2016 absolute change",
                "2014–2016 percentage change", "Highest 2016 count", "Highest 2016 rate",
                "Largest 2014–2016 increase", "Largest 2014–2016 decrease", "Population-rate Pearson r"],
    "result": [
        f"{yearly_totals.loc[2014]:,.0f}", f"{yearly_totals.loc[2015]:,.0f}", f"{yearly_totals.loc[2016]:,.0f}",
        f"{overall_absolute_change:+,}", f"{overall_percentage_change:+.2f}%",
        f"{crime.loc[crime.crimes_2016.idxmax(), 'state_ut']} ({crime.crimes_2016.max():,.0f})",
        f"{crime.loc[crime.crime_rate_2016.idxmax(), 'state_ut']} ({crime.crime_rate_2016.max():.1f} per 100,000)",
        f"{regional_changes.loc[regional_changes.absolute_change.idxmax(), 'state_ut']} (+{regional_changes.absolute_change.max():,.0f}; {regional_changes.loc[regional_changes.absolute_change.idxmax(), 'percentage_change']:+.2f}%)",
        f"{regional_changes.loc[regional_changes.absolute_change.idxmin(), 'state_ut']} ({regional_changes.absolute_change.min():,.0f}; {regional_changes.loc[regional_changes.absolute_change.idxmin(), 'percentage_change']:+.2f}%)",
        f"{pearson_r:.3f}",
    ],
})
display(observations)

### Interpretation and limitations

**Observed:** Recorded totals rose from 2014 to 2016, with a smaller percentage increase from 2015 to 2016 than from 2014 to 2015. Uttar Pradesh had the highest 2016 count; Delhi had the highest 2016 rate. These identify different aspects of the data because counts and population-adjusted rates answer different questions. Delhi and Kerala are beyond the overall rate box plot’s upper 1.5-IQR fence; this does not establish that their values are wrong.

**Limitations:** Three annual observations are insufficient to establish a durable trend. Recorded-crime data depend on reporting and classification. State and UT groups have unequal sample sizes and are not population-weighted. The correlation is descriptive and does not explain causes.

**Future questions:** How do patterns compare with later years under comparable definitions? Do regional comparisons change when rates are tracked over time? How do reporting and demographic differences affect these comparisons?